## **Fake news Classifier**
A fake news detection system developed using DistilBERT to classify news articles as real or fake. The project uses the ISOT Fake News Dataset and explores transformer-based text classification for identifying patterns associated with misleading and reliable news articles.

The workflow includes :

*   dataset preparation
*   cleaning and preprocessing of news articles


*   tokenization using the DistilBERT tokenizer
*   token-based chunking to handle articles that exceed the model's maximum input length



 Since a long article can produce multiple chunks, each chunk is classified individually and the predictions are then aggregated to generate a single article-level prediction.

To make training more computationally efficient, the project uses partial *fine-tuning by freezing the earlier DistilBERT transformer layers* while fine-tuning the later layers and classification head.


In [60]:
!pip install -q kaggle

In [ ]:
import os

os.environ["KAGGLE_API_TOKEN"] = "KAGGLE_API_KEY"

In [70]:
!kaggle datasets list -s "ISOT Fake News"

ref                                             title                                      size  lastUpdated                 downloadCount  voteCount  usabilityRating  
----------------------------------------------  -----------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
clmentbisaillon/fake-and-real-news-dataset      fake-and-real-news-dataset             42975911  2024-04-19 13:02:33.387000         119710        433                1  
rahulogoel/isot-fake-news-dataset               ISOT Fake News Dataset                 42975963  2024-12-29 07:42:07.230000           3183         14        0.7647059  
mucahiddemircan/real-and-fake-news-dataset      Real and Fake News Dataset             48992264  2026-05-26 15:36:30.117000           1140         34                1  
emineyetm/fake-news-detection-datasets          Fake News Detection Datasets           42975967  2022-12-07 11:45:17.723000          44381        156      

In [62]:
!kaggle datasets download -d rahulogoel/isot-fake-news-dataset

Dataset URL: https://www.kaggle.com/datasets/rahulogoel/isot-fake-news-dataset
License(s): MIT
isot-fake-news-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)


In [63]:
import zipfile

with zipfile.ZipFile("isot-fake-news-dataset.zip", "r") as zip_ref:
    zip_ref.extractall("isot_data")

In [64]:
import os

print(os.listdir("isot_data"))

['News_Dataset']


In [65]:
import os

print("Current directory:")
print(os.getcwd())

print("\nFiles/folders here:")
print(os.listdir())

print("\nSearching for CSV files:")
for root, dirs, files in os.walk("."):
    for file in files:
        if file.endswith(".csv"):
            print(os.path.join(root, file))

Current directory:
/content

Files/folders here:
['.config', 'fake_news_distilbert', 'isot-fake-news-dataset.zip', 'isot_data', 'sample_data']

Searching for CSV files:
./isot_data/News_Dataset/True.csv
./isot_data/News_Dataset/Fake.csv
./sample_data/mnist_test.csv
./sample_data/california_housing_train.csv
./sample_data/mnist_train_small.csv
./sample_data/california_housing_test.csv


In [66]:
import pandas as pd

true_news = pd.read_csv("isot_data/News_Dataset/True.csv")
fake_news = pd.read_csv("isot_data/News_Dataset/Fake.csv")

print("Real news:", true_news.shape)
print("Fake news:", fake_news.shape)

Real news: (21417, 4)
Fake news: (23481, 4)


In [68]:
true_news["label"] = 0
fake_news["label"] = 1

news = pd.concat([true_news, fake_news], ignore_index=True)

print("Dataset shape: ", news.shape)
print("Label distribution")
print(news["label"].value_counts())

Dataset shape:  (44898, 5)
Label distribution
label
1    23481
0    21417
Name: count, dtype: int64


In [71]:
# shuffle the dataset

news = news.sample(frac=1, random_state=42).reset_index(drop=True)

In [72]:
display(news.head())

,title,text,subject,date,label
0,BREAKING: GOP Chairman Grassley Has Had Enoug...,"Donald Trump s White House is in chaos, and th...",News,"July 21, 2017",1
1,Failed GOP Candidates Remembered In Hilarious...,Now that Donald Trump is the presumptive GOP n...,News,"May 7, 2016",1
2,Mike Pence’s New DC Neighbors Are HILARIOUSLY...,Mike Pence is a huge homophobe. He supports ex...,News,"December 3, 2016",1
3,California AG pledges to defend birth control ...,SAN FRANCISCO (Reuters) - California Attorney ...,politicsNews,"October 6, 2017",0
4,AZ RANCHERS Living On US-Mexico Border Destroy...,Twisted reasoning is all that comes from Pelos...,politics,"Apr 25, 2017",1


In [73]:
print("Missing values:")
print(news.isnull().sum())

print("\nDuplicate rows:")
print(news.duplicated().sum())

print("\nDuplicate texts:")
print(news["text"].duplicated().sum())

print("\nLabel distribution:")
print(news["label"].value_counts())

Missing values:
title      0
text       0
subject    0
date       0
label      0
dtype: int64

Duplicate rows:
209

Duplicate texts:
6252

Label distribution:
label
1    23481
0    21417
Name: count, dtype: int64


In [74]:
news = news.drop_duplicates(subset=["text"]).reset_index(drop=True)

print("Duplicate shape after removing duplicate texts:", news.shape)
print("Label Distribution: ")
print(news["label"].value_counts())

Duplicate shape after removing duplicate texts: (38646, 5)
Label Distribution: 
label
0    21191
1    17455
Name: count, dtype: int64


In [75]:
news["text_length"] = news["text"].str.len()
news["word_count"] = news["text"].str.split().str.len()

print("Character length")
print(news["text_length"].describe())

print("============================")

print("Word count: ")
print(news["word_count"].describe())

Character length
count    38646.000000
mean      2455.888061
std       1936.609582
min          1.000000
25%       1318.000000
50%       2227.000000
75%       3096.000000
max      51794.000000
Name: text_length, dtype: float64
Word count: 
count    38646.000000
mean       403.089971
std        313.828131
min          0.000000
25%        216.000000
50%        369.000000
75%        511.000000
max       8135.000000
Name: word_count, dtype: float64


In [77]:
news[news["word_count"] < 5][["title", "text", "label"]]

,title,text,label
19,FATHER OF BENGHAZI VICTIM TY WOODS SPEAKS UP A...,,1
70,PELOSI Lied About “Affordable” Healthcare And ...,Via: WT,1
965,WOW! TUCKER AND JESSE Destroy The Liberal Kook...,https://www.youtube.com/watch?v=J4LjxrOfEF8,1
984,REPORTER ASKS: Will Obama Golf Instead of Atte...,BRAVE GUY!,1
1132,‘YOU’RE A SEXIST PIG!’ FEMINIST BATTLES Tucker...,https://www.youtube.com/watch?v=YeDU6dCR9tA,1
...,...,...,...
34934,TERRORISM EXPERT SEBASTIAN GORKA On What Next ...,https://www.youtube.com/watch?v=-7Tn4gi_Os8,1
34999,WATCH! TRUMP SUPPORTER “BIG JOE” Surrounded By...,https://www.youtube.com/watch?v=IPqrimR8GWw,1
35907,KELLYANNE CONWAY On Trump’s Terrorism Policy: ...,https://www.youtube.com/watch?v=0cVugq2GbBk,1
36114,VIRAL VIDEO UPDATE: MARINE VET Has To Be Evacu...,UPDATE VIDEO: Language alert:,1


In [78]:
news = news[news["word_count"] >= 5].reset_index(drop=True)

print("Dataset shape:", news.shape)
print("Label Distribution: ")
print(news["label"].value_counts())

Dataset shape: (38553, 7)
Label Distribution: 
label
0    21191
1    17362
Name: count, dtype: int64


In [79]:
news = news.drop(columns=["text_length", "word_count"])

In [80]:
import torch
import numpy as np
import pandas as pd

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)

In [ ]:
# select 1000 articles
subset = news.sample(
    n=1000,
    random_state=42
).reset_index(drop=True)

print("Dataset size:", len(subset))
print(subset["label"].value_counts())

In [ ]:
# train test validation split
train_df, temp_df = train_test_split(
    subset,
    test_size=0.30,
    stratify=subset["label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

In [ ]:
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

In [ ]:
train_df["article_id"] = train_df.index
val_df["article_id"] = val_df.index
test_df["article_id"] = test_df.index

In [ ]:
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

In [ ]:
# load DistilBERT tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased"
)

In [ ]:
'''
  480 original tokens
  64 token overlap
  special tokens: [CLS] and [SEP] added afterward
'''
def chunk_text(text, chunk_size=480, overlap=64):

    tokens = tokenizer(
        text,
        add_special_tokens=False,
        truncation=False
    )["input_ids"]

    chunks = []
    start = 0

    while start < len(tokens):

        end = start + chunk_size

        chunks.append(tokens[start:end])

        if end >= len(tokens):
            break

        start = end - overlap

    return chunks

In [ ]:
# prepare each chunk
def prepare_chunk(chunk):

    input_ids = (
        [tokenizer.cls_token_id]
        + chunk
        + [tokenizer.sep_token_id]
    )

    attention_mask = [1] * len(input_ids)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask
    }

In [ ]:
# convert articles into chunk records
def chunk_articles(batch):

    all_article_ids = []
    all_input_ids = []
    all_attention_masks = []
    all_labels = []

    for article_id, text, label in zip(
        batch["article_id"],
        batch["text"],
        batch["label"]
    ):

        chunks = chunk_text(text)

        for chunk in chunks:

            prepared = prepare_chunk(chunk)

            all_article_ids.append(article_id)
            all_input_ids.append(prepared["input_ids"])
            all_attention_masks.append(
                prepared["attention_mask"]
            )
            all_labels.append(label)

    return {
        "article_id": all_article_ids,
        "input_ids": all_input_ids,
        "attention_mask": all_attention_masks,
        "labels": all_labels
    }

In [ ]:
# convert dataframes into hugging face datasets
train_articles = Dataset.from_pandas(
    train_df[["article_id", "text", "label"]],
    preserve_index=False
)

val_articles = Dataset.from_pandas(
    val_df[["article_id", "text", "label"]],
    preserve_index=False
)

test_articles = Dataset.from_pandas(
    test_df[["article_id", "text", "label"]],
    preserve_index=False
)

In [ ]:
# create chunk datasets
train_chunk_dataset = train_articles.map(
    chunk_articles,
    batched=True,
    batch_size=100,
    remove_columns=["text", "label"]
)

val_chunk_dataset = val_articles.map(
    chunk_articles,
    batched=True,
    batch_size=100,
    remove_columns=["text", "label"]
)

test_chunk_dataset = test_articles.map(
    chunk_articles,
    batched=True,
    batch_size=100,
    remove_columns=["text", "label"]
)

In [ ]:
print("Train chunks:", len(train_chunk_dataset))
print("Validation chunks:", len(val_chunk_dataset))
print("Test chunks:", len(test_chunk_dataset))

In [ ]:
# Dynamic Padding
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

In [ ]:
# freeze first 4 layers
for layer in model.distilbert.transformer.layer[:4]:

    for param in layer.parameters():
        param.requires_grad = False

# verifying
for i, layer in enumerate(model.distilbert.transformer.layer):

    print(
        f"Layer {i+1}:",
        any(p.requires_grad for p in layer.parameters())
    )

print(
    "Classifier:",
    any(p.requires_grad for p in model.classifier.parameters())
)

In [ ]:
# chunk-level metrics
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="binary"
        )
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="./fake_news_distilbert",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_chunk_dataset,
    eval_dataset=val_chunk_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

In [ ]:
# predictions
pred_output = trainer.predict(
    test_chunk_dataset
)


In [ ]:
logits = pred_output.predictions

chunk_probs = torch.softmax(
    torch.tensor(logits),
    dim=1
).numpy()[:, 1]

chunk_labels = pred_output.label_ids

article_ids = np.array(
    test_chunk_dataset["article_id"]
)

test_predictions = pd.DataFrame({
    "article_id": article_ids,
    "probability": chunk_probs,
    "label": chunk_labels
})

In [ ]:
article_predictions = (
    test_predictions
    .groupby("article_id")
    .agg({
        "probability": "mean",
        "label": "first"
    })
    .reset_index()
)

article_predictions["prediction"] = (
    article_predictions["probability"] >= 0.5
).astype(int)

In [69]:
# final metrics
accuracy = accuracy_score(
    article_predictions["label"],
    article_predictions["prediction"]
)

precision, recall, f1, _ = (
    precision_recall_fscore_support(
        article_predictions["label"],
        article_predictions["prediction"],
        average="binary"
    )
)

print("Article-level TEST results")
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1       :", f1)

Article-level TEST results
Accuracy : 0.98
Precision: 0.971830985915493
Recall   : 0.9857142857142858
F1       : 0.9787234042553191


In [ ]:
cm = confusion_matrix(
    article_predictions["label"],
    article_predictions["prediction"]
)

print("\nConfusion Matrix:")
print(cm)